In [5]:
import numpy as np
from numba import cuda
import math

BLOCK_SIZE = 256

@cuda.jit
def memory_management_kernel(d_input, d_output):
    # 1. SHARED MEMORY ALLOCATION
    s_block_data = cuda.shared.array(shape=BLOCK_SIZE, dtype=cuda.float32)

    # 2. LOCAL MEMORY ALLOCATION
    tx = cuda.threadIdx.x
    bx = cuda.blockIdx.x
    bdim = cuda.blockDim.x
    global_idx = bx * bdim + tx


    # Load from Global Memory into Shared Memory ---
    if global_idx < d_input.size:
        s_block_data[tx] = d_input[global_idx]
    else:
        s_block_data[tx] = 0.0

    # BARRIER SYNCHRONIZATION:
    cuda.syncthreads()

    #  Manipulate Data Using Shared & Local Memory
    reversed_tx = (bdim - 1) - tx

    # --- STEP C: Write Results Back to Global Memory ---
    if global_idx < d_output.size:
        d_output[global_idx] = s_block_data[reversed_tx]

def run_memory_demo(h_array):
    n = h_array.size
    threads_per_block = BLOCK_SIZE
    blocks_per_grid = math.ceil(n / threads_per_block)

    # Allocating Global Memory on the Device (VRAM)
    d_input = cuda.to_device(h_array.astype(np.float32))
    d_output = cuda.device_array(n, dtype=np.float32)

    # Launching kernel
    memory_management_kernel[blocks_per_grid, threads_per_block](d_input, d_output)

    return d_output.copy_to_host()

if __name__ == "__main__":
    # Create an array of 512 elements
    array_size = 512
    test_data = np.arange(array_size, dtype=np.float32)

    print("Original Global Data Chunks (First 10 elements of block 1):")
    print(test_data[:10])
    print("Original Global Data Chunks (First 10 elements of block 2):")
    print(test_data[256:266])
    print("-" * 70)

    # Run the memory managed kernel
    result_data = run_memory_demo(test_data)

    print("Processed Global Data Output (First 10 elements of block 1 - Reversed):")
    print(result_data[:10])
    print("Processed Global Data Output (First 10 elements of block 2 - Reversed):")
    print(result_data[256:266])

Original Global Data Chunks (First 10 elements of block 1):
[0. 1. 2. 3. 4. 5. 6. 7. 8. 9.]
Original Global Data Chunks (First 10 elements of block 2):
[256. 257. 258. 259. 260. 261. 262. 263. 264. 265.]
----------------------------------------------------------------------
Processed Global Data Output (First 10 elements of block 1 - Reversed):
[255. 254. 253. 252. 251. 250. 249. 248. 247. 246.]
Processed Global Data Output (First 10 elements of block 2 - Reversed):
[511. 510. 509. 508. 507. 506. 505. 504. 503. 502.]


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:748: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
